In [17]:
from scipy.stats import kendalltau
import numpy as np
import pandas as pd
import copy
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def _fftfreq_radius(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    grids = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(sum(g**2 for g in grids))
    return r


def _fsc(vol1, vol2, voxel_size=1.0, df=0.01, eps=1e-12):
    """Compute Fourier shell correlation between two 3D arrays."""
    assert vol1.shape == vol2.shape
    vol1 = vol1 - np.mean(vol1)
    vol2 = vol2 - np.mean(vol2)

    F1 = np.fft.fftn(vol1)
    F2 = np.fft.fftn(vol2)
    r = _fftfreq_radius(vol1.shape, voxel_size)

    r_max = r.max()
    edges = np.arange(0, r_max + df, df)
    bins = np.digitize(r.ravel(), edges) - 1
    n_shells = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n_shells)
    b = bins[valid]
    num = np.bincount(b, weights=cross[valid], minlength=n_shells)
    den1 = np.bincount(b, weights=p1[valid], minlength=n_shells)
    den2 = np.bincount(b, weights=p2[valid], minlength=n_shells)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

def fsc_resolution(freqs, fsc, threshold=0.143):
    """Linear interpolation to find 1/f at threshold crossing."""
    below = np.where(fsc < threshold)[0]
    if not len(below):
        return 1.0
    i = below[0]
    if i == 0:
        f_cut = freqs[i]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc[i-1], fsc[i]
        f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)
    return 1.0 / f_cut

def subsect_sinogram(sino, time_index, window_end):
    sino_copy = copy.deepcopy(sino)
    sino_copy.data = sino.data[time_index:window_end, :, :]
    sino_copy.times = sino.times[time_index:window_end]
    sino_copy.angles = sino.angles[time_index:window_end]
    return sino_copy


def calculate_fsc(sino, recon):
    sino_half1 = subsect_sinogram(sino, 0, len(sino.times)//2)
    sino_half2 = subsect_sinogram(sino, len(sino.times)//2, len(sino.times))

    recon_half1 = recon(sino_half1)
    recon_half2 = recon(sino_half2)
    freqs, fsc = _fsc(recon_half1.data, recon_half2.data)
    res = fsc_resolution(freqs, fsc)

    del sino_half1, sino_half2, recon_half1, recon_half2
    return res


def calculate_window_fsc(sino, recon, time_index, window_end):
    sino_sub = subsect_sinogram(sino, time_index, window_end)
    fsc_value = calculate_fsc(sino_sub, recon)
    del sino_sub
    return fsc_value

def dynamic_reconstruction(sino, recon, rolling_window=3, initial_window=21, step=1, max_rw=15, *args, **kwargs):
    windows  = np.zeros((len(sino.times),3), dtype=int)
    x = (rolling_window*2)+1
    
    for i in range(len(sino.times)):
        windows[i,0] = i
        windows[i,1] = i+initial_window
        windows[i,2] = 1

    for i, t in enumerate(sino.times):
        if windows[i,1] > (len(sino.times)-1):
            windows[i:, 2] = 0
            break


        break_out = False
        set_values = False
        direction = 0
        rolling_values = np.full(x, np.nan)

        print(f'\n[{windows[i,0]}]---- Estimated {windows[i,1]-windows[i,0]}-----------------------------------------------------')
        df = pd.DataFrame(columns=['Windows', 'Window Size', 'Time (s)', 'FSC', 'FSC Std', 'Kendall Tau', 'p-value', 'Direction'])
        while not break_out:
            print(f'[{windows[i,0]}|{windows[i,1]}] Measuring FSC at window size: {rolling_window} ')
            if set_values:
                rolling_values = np.roll(rolling_values, -1*direction)
                if direction == 1:
                    rolling_values[-1] = calculate_window_fsc(sino, recon, i, windows[i,1]+rolling_window)
                elif direction == -1:
                    rolling_values[0] = calculate_window_fsc(sino, recon, i, windows[i,1]-rolling_window)
            else:
                for j, k in enumerate(range(windows[i,1]-rolling_window, windows[i,1]+rolling_window+1)):
                    print(j, k, windows[i,0], windows[i,1])
                    if k >= (len(sino.times)) or k <= windows[i,0]+1:
                        rolling_values[j] = np.nan
                    else:
                        rolling_values[j] = calculate_window_fsc(sino, recon, i,  k)
                    
                set_values = True

            cropped_values = rolling_values[~np.isnan(rolling_values)]
            tau, p_value = kendalltau(np.arange(len(cropped_values)), cropped_values)

            while p_value > 0.05 and max_rw > rolling_window:
                rolling_window += 1
                x += 2
                concat_array = np.full(2, np.nan)
                rolling_values = np.concatenate((rolling_values, concat_array))
                rolling_values = np.roll(rolling_values, 1)
                rolling_values[-1] = calculate_window_fsc(sino, recon, i, windows[i,1]+rolling_window)
                if windows[i,1]-rolling_window >= windows[i,0]:
                    rolling_values[0] = np.nan
                else:
                    rolling_values[0] = calculate_window_fsc(sino, recon, i, windows[i,1]-rolling_window)

                
                cropped_values = rolling_values[~np.isnan(rolling_values)]
                tau, p_value = kendalltau(np.arange(len(cropped_values)), cropped_values)
                print(f'[{windows[i,0]}|{windows[i,1]}] Increased measurement window to {rolling_window} due to insignificant -> p-value: {p_value}')
                
            if p_value < 0.01:
                x -= 2
                rolling_window -= 1
                rolling_values = rolling_values[1:-1]
                print(f'[{windows[i,0]}|{windows[i,1]}] Decreasing measurement window to {rolling_window} due to computational time -> p-value: {p_value}')

            if tau < 0 and np.abs(tau) > 0.1:
                new_direction = 1
            elif tau > 0 and np.abs(tau) > 0.1:
                new_direction = -1 
            else:
                new_direction = 0

            df_new = pd.DataFrame([{
                    'Windows': f'[{windows[i,0]}|{windows[i,1]}]',
                    'Window Size': windows[i,1]-windows[i,0],
                    'Time (s)': np.round(np.mean(sino.times[windows[i,0]:windows[i,1]]),2),
                    'FSC': rolling_values[rolling_window],
                    'FSC Std': np.std(rolling_values),
                    'Kendall Tau': tau,
                    'p-value': p_value, 
                    'Direction': new_direction
                }])
            
            df = pd.concat([df, df_new], ignore_index=True)

            if new_direction == 1 and windows[i,1] >= (len(sino.times)-1):
                new_direction = 0
            elif new_direction == -1 and windows[i,1] <= (windows[i,0] + rolling_window):
                new_direction = 0

            if ((direction + new_direction) == 0) or new_direction == 0:
                windows[i+1,1] = (windows[i,1] - windows[i,0]) + 1 + windows[i,0]
                break_out = True
                print(df)

            direction = new_direction
            if direction == 1:
                windows[i,1] += step
            elif direction == -1:
                windows[i,1] -= step

    return windows


In [18]:
from tomobase.data import Sinogram, Volume
from tomobase import processes
from functools import partial

astra = partial(processes.astra_reconstruct, method='SIRT', iterations=100)
sino = Sinogram.from_file(r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\TiltSeries128\Cage-D.mrc')
print(sino.data.shape, len(sino.times), sino.angles.shape)
windows = dynamic_reconstruction(sino, astra, rolling_window=3, initial_window=21, step=2, max_rw=15)

(100, 128, 128) 100 (100,)

[0]---- Estimated 21-----------------------------------------------------
[0|21] Measuring FSC at window size: 3 
0 18 0 21
1 19 0 21
2 20 0 21
3 21 0 21
4 22 0 21
5 23 0 21
6 24 0 21


C:\Users\TCraig\AppData\Local\Temp\ipykernel_53240\82413318.py:170: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_new], ignore_index=True)


[0|19] Measuring FSC at window size: 3 
[0|17] Measuring FSC at window size: 3 
[0|15] Measuring FSC at window size: 3 
[0|13] Measuring FSC at window size: 3 
[0|13] Decreasing measurement window to 2 due to computational time -> p-value: 0.002777777777777778
[0|11] Measuring FSC at window size: 2 
[0|11] Increased measurement window to 3 due to insignificant -> p-value: 0.12597116307723114
[0|11] Increased measurement window to 4 due to insignificant -> p-value: 0.12873504895794796
[0|11] Increased measurement window to 5 due to insignificant -> p-value: 0.04437842734548687
[0|9] Measuring FSC at window size: 5 
[0|9] Increased measurement window to 6 due to insignificant -> p-value: 0.6732899796599957
[0|9] Increased measurement window to 7 due to insignificant -> p-value: 0.3652760444092834
[0|9] Increased measurement window to 8 due to insignificant -> p-value: 0.15729920705028502
[0|9] Increased measurement window to 9 due to insignificant -> p-value: 0.16619516939049406
[0|9] In

C:\Users\TCraig\AppData\Local\Temp\ipykernel_53240\82413318.py:170: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_new], ignore_index=True)


[1|12] Measuring FSC at window size: 15 
[1|14] Measuring FSC at window size: 15 
[1|16] Measuring FSC at window size: 15 
[1|18] Measuring FSC at window size: 15 
[1|20] Measuring FSC at window size: 15 
[1|22] Measuring FSC at window size: 15 
[1|24] Measuring FSC at window size: 15 
[1|26] Measuring FSC at window size: 15 
[1|28] Measuring FSC at window size: 15 
  Windows Window Size  Time (s)       FSC    FSC Std  Kendall Tau   p-value Direction
0  [1|10]           9       6.0  2.599416        NaN    -0.383399  0.010075         1
1  [1|12]          11       7.0  2.161966        NaN    -0.340580  0.019868         1
2  [1|14]          13       8.0  2.379788        NaN    -0.300000  0.036674         1
3  [1|16]          15       9.0  2.143696        NaN    -0.255385  0.070392         1
4  [1|18]          17      10.0  2.427195        NaN    -0.219373  0.113682         1
5  [1|20]          19      11.0  2.452341        NaN    -0.206349  0.129029         1
6  [1|22]          21      12

...

   Windows Window Size  Time (s)       FSC  FSC Std  Kendall Tau   p-value Direction
0  [30|89]          59      60.0  1.727569      NaN    -0.009231  0.965203         0

[31]---- Estimated 59-----------------------------------------------------
[31|90] Measuring FSC at window size: 15 
0 75 31 90
1 76 31 90
2 77 31 90
3 78 31 90
4 79 31 90
5 80 31 90
6 81 31 90
7 82 31 90
8 83 31 90
9 84 31 90
10 85 31 90
11 86 31 90
12 87 31 90
13 88 31 90
14 89 31 90
15 90 31 90
16 91 31 90
17 92 31 90
18 93 31 90
19 94 31 90
20 95 31 90
21 96 31 90
22 97 31 90
23 98 31 90
24 99 31 90
25 100 31 90
26 101 31 90
27 102 31 90
28 103 31 90
29 104 31 90
30 105 31 90


C:\Users\TCraig\AppData\Local\Temp\ipykernel_53240\82413318.py:170: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_new], ignore_index=True)


   Windows Window Size  Time (s)       FSC  FSC Std  Kendall Tau   p-value Direction
0  [31|90]          59      61.0  1.723605      NaN        -0.02  0.908033         0

[32]---- Estimated 59-----------------------------------------------------
[32|91] Measuring FSC at window size: 15 
0 76 32 91
1 77 32 91
2 78 32 91
3 79 32 91
4 80 32 91
5 81 32 91
6 82 32 91
7 83 32 91
8 84 32 91
9 85 32 91
10 86 32 91
11 87 32 91
12 88 32 91
13 89 32 91
14 90 32 91
15 91 32 91
16 92 32 91
17 93 32 91
18 94 32 91
19 95 32 91
20 96 32 91
21 97 32 91
22 98 32 91
23 99 32 91
24 100 32 91
25 101 32 91
26 102 32 91
27 103 32 91
28 104 32 91
29 105 32 91
30 106 32 91


C:\Users\TCraig\AppData\Local\Temp\ipykernel_53240\82413318.py:170: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_new], ignore_index=True)


[32|93] Measuring FSC at window size: 15 
[32|95] Measuring FSC at window size: 15 
[32|95] Decreasing measurement window to 14 due to computational time -> p-value: 0.0016264080109989576
[32|97] Measuring FSC at window size: 14 
[32|97] Decreasing measurement window to 13 due to computational time -> p-value: 0.005701073711101261
[32|99] Measuring FSC at window size: 13 
[32|99] Increased measurement window to 14 due to insignificant -> p-value: 0.023294806349832088
   Windows Window Size  Time (s)       FSC  FSC Std  Kendall Tau   p-value Direction
0  [32|91]          59      62.0  1.724397      NaN    -0.159420  0.289552         1
1  [32|93]          61      63.0  1.716150      NaN    -0.304348  0.038522         1
2  [32|95]          63      64.0  1.719318      NaN    -0.460981  0.001626         1
3  [32|97]          65      65.0  1.733274      NaN    -0.425164  0.005701         1
4  [32|99]          67      66.0  1.732527      NaN    -0.359722  0.023295         1


In [20]:
for i in range(windows.shape[0]):
    if windows[i,2] == 1:
        print(f'Window {i}: Start: {windows[i,0]} | End: {windows[i,1]} | Size: {windows[i,1]-windows[i,0]} Time (s): {np.mean(sino.times[windows[i,0]:windows[i,1]])}')

Window 0: Start: 0 | End: 9 | Size: 9 Time (s): 5.0
Window 1: Start: 1 | End: 28 | Size: 27 Time (s): 15.0
Window 2: Start: 2 | End: 21 | Size: 19 Time (s): 12.0
Window 3: Start: 3 | End: 26 | Size: 23 Time (s): 15.0
Window 4: Start: 4 | End: 25 | Size: 21 Time (s): 15.0
Window 5: Start: 5 | End: 32 | Size: 27 Time (s): 19.0
Window 6: Start: 6 | End: 39 | Size: 33 Time (s): 23.0
Window 7: Start: 7 | End: 28 | Size: 21 Time (s): 18.0
Window 8: Start: 8 | End: 35 | Size: 27 Time (s): 22.0
Window 9: Start: 9 | End: 40 | Size: 31 Time (s): 25.0
Window 10: Start: 10 | End: 37 | Size: 27 Time (s): 24.0
Window 11: Start: 11 | End: 44 | Size: 33 Time (s): 28.0
Window 12: Start: 12 | End: 47 | Size: 35 Time (s): 30.0
Window 13: Start: 13 | End: 46 | Size: 33 Time (s): 30.0
Window 14: Start: 14 | End: 47 | Size: 33 Time (s): 31.0
Window 15: Start: 15 | End: 48 | Size: 33 Time (s): 32.0
Window 16: Start: 16 | End: 49 | Size: 33 Time (s): 33.0
Window 17: Start: 17 | End: 88 | Size: 71 Time (s): 53